# 02 - Signal Research

## Purpose of this notebook

In this notebook, I build the main signals that the strategy will use.

The goal here is to transform the cleaned ETF price data into decision variables for the portfolio model. In particular, I want to compute:

- momentum signals over different lookback windows
- a 200-day moving average trend filter
- rolling volatility estimates
- a month-end aligned signal table that can later feed into the backtest

This notebook is important because it connects the data pipeline to the strategy logic. By the end of this notebook, I want to know which ETFs look strong, which ETFs pass the long-term trend filter, and how risky each ETF currently appears.

## Main research idea

The strategy family I am testing is a tactical multi-ETF allocation strategy based on:

- momentum ranking
- 200-day moving average filtering
- top-K selection
- volatility-aware weighting

At this stage, I am not yet testing final portfolio performance. I am first making sure the signals are computed correctly and aligned properly.

## Data split used in this project

To make the project more rigorous, I split the data into three periods:

- **Train:** 2005-01-01 to 2014-12-31
- **Validation:** 2015-01-01 to 2018-12-31
- **Test:** 2019-01-01 onward

This lets me develop and compare strategy ideas without using the final test period too early.

## 2. Import libraries

In this section, I import the libraries needed for signal construction and analysis.

### What this section is doing
These imports let me:
- load and manipulate time series data
- compute signal transformations
- work with project file paths
- create charts for sanity-checking the signals

### What I can change later
- I can add more plotting libraries later if I want more polished charts
- I can move reusable functions into `src/` later once they are stable

### What to expect
This section should run quietly unless a package is missing.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True

## 3. Load processed data

In this section, I load the cleaned datasets created in Notebook 01.

### What this section is doing
I load:
- daily adjusted close prices
- monthly prices
- monthly returns

These files will be used to construct the strategy signals.

### Why this matters
The strategy uses both daily and monthly information:
- daily prices are needed for the 200-day moving average and rolling volatility
- monthly prices and returns are useful for lower-turnover momentum research

### What I can change later
- If I save additional processed features later, I can load them here too
- If I switch to Parquet files later, I can update the read logic

### What to expect
I should see clean dataframes with datetime indexes and ETF columns.

In [2]:
DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"

daily_adj_close = pd.read_csv(PROCESSED_DIR / "daily_adj_close.csv", index_col=0, parse_dates=True)
monthly_prices = pd.read_csv(PROCESSED_DIR / "monthly_prices.csv", index_col=0, parse_dates=True)
monthly_returns = pd.read_csv(PROCESSED_DIR / "monthly_returns.csv", index_col=0, parse_dates=True)

print("Daily adjusted close shape:", daily_adj_close.shape)
print("Monthly prices shape:", monthly_prices.shape)
print("Monthly returns shape:", monthly_returns.shape)

display(daily_adj_close.head())
display(monthly_prices.head())
display(monthly_returns.head())

Daily adjusted close shape: (5332, 8)
Monthly prices shape: (254, 8)
Monthly returns shape: (253, 8)


,EEM,EFA,GLD,IEF,IWM,QQQ,SPY,TLT
Date,,,,,,,,
2005-01-03,14.562677,28.951866,43.020000,48.410713,48.275024,33.695381,81.606010,44.738453
2005-01-04,14.114314,28.396852,42.740002,48.109287,47.240181,33.081181,80.608803,44.269611
2005-01-05,13.941525,28.378651,42.669998,48.188850,46.295994,32.876431,80.052544,44.506535
2005-01-06,13.932051,28.378651,42.150002,48.234386,46.541466,32.714378,80.459557,44.536804
2005-01-07,13.959026,28.251270,41.840000,48.194592,46.024055,32.884983,80.344238,44.637630


,EEM,EFA,GLD,IEF,IWM,QQQ,SPY,TLT
Date,,,,,,,,
2005-01-31,14.637771,28.606121,42.220001,48.734882,46.922932,31.903990,80.154320,46.235783
2005-02-28,16.055035,29.688854,43.529999,48.067986,47.693409,31.750433,81.829826,45.555168
2005-03-31,14.785039,28.910019,42.820000,47.874046,46.341408,31.195953,80.332947,45.348000
2005-04-29,14.600583,28.442339,43.349998,49.078831,43.719868,29.839607,78.827843,47.100842
2005-05-31,15.062075,28.196678,41.650002,49.981003,46.546272,32.484039,81.368103,48.578629


,EEM,EFA,GLD,IEF,IWM,QQQ,SPY,TLT
Date,,,,,,,,
2005-02-28,0.096822,0.037850,0.031028,-0.013684,0.016420,-0.004813,0.020904,-0.014721
2005-03-31,-0.079103,-0.026233,-0.016311,-0.004035,-0.028348,-0.017464,-0.018293,-0.004548
2005-04-29,-0.012476,-0.016177,0.012377,0.025166,-0.056570,-0.043478,-0.018736,0.038653
2005-05-31,0.031608,-0.008637,-0.039216,0.018382,0.064648,0.088622,0.032225,0.031375
2005-06-30,0.039690,0.014327,0.042977,0.004784,0.040763,-0.033246,0.001515,0.021648


## 4. Define the train / validation / test split

In this section, I define the date ranges that will be used for model development and evaluation.

### What this section is doing
I split the data into three periods:
- train
- validation
- test

### Why this matters
This is one of the most important parts of making the project rigorous.

Instead of tuning everything on the full sample, I will:
- develop candidate strategies on the train period
- compare and select on the validation period
- reserve the test period for final out-of-sample evaluation

### What I can change later
- I can adjust the split if I want a different research design
- I can later add rolling walk-forward evaluation in addition to this fixed split

### What to expect
I should see the date ranges printed clearly, along with the number of observations in each split.

In [3]:
TRAIN_START = "2005-01-01"
TRAIN_END = "2014-12-31"

VALID_START = "2015-01-01"
VALID_END = "2018-12-31"

TEST_START = "2019-01-01"
TEST_END = monthly_prices.index.max().strftime("%Y-%m-%d")

print("Train:", TRAIN_START, "to", TRAIN_END)
print("Validation:", VALID_START, "to", VALID_END)
print("Test:", TEST_START, "to", TEST_END)

Train: 2005-01-01 to 2014-12-31
Validation: 2015-01-01 to 2018-12-31
Test: 2019-01-01 to 2026-02-27


In [4]:
def split_by_date(df, train_start, train_end, valid_start, valid_end, test_start, test_end):
    train = df.loc[train_start:train_end].copy()
    valid = df.loc[valid_start:valid_end].copy()
    test = df.loc[test_start:test_end].copy()
    return train, valid, test

monthly_prices_train, monthly_prices_valid, monthly_prices_test = split_by_date(
    monthly_prices, TRAIN_START, TRAIN_END, VALID_START, VALID_END, TEST_START, TEST_END
)

print("Monthly prices observations")
print("Train:", len(monthly_prices_train))
print("Validation:", len(monthly_prices_valid))
print("Test:", len(monthly_prices_test))

Monthly prices observations
Train: 120
Validation: 48
Test: 86


## 5. Baseline strategy design

Before calculating any signals, I write down the baseline strategy in plain English.

### Current baseline idea
At each monthly rebalance date, the strategy will:

1. compute momentum scores for all ETFs
2. check whether each ETF is above its 200-day moving average
3. rank ETFs by momentum
4. keep only ETFs that pass the moving average filter
5. select the top K ETFs
6. later assign portfolio weights using a volatility-based method

### Why I am writing this down
This helps keep the project disciplined. If the strategy rules stay clear in plain English, it is much easier to:
- check whether the code matches the idea
- avoid accidental look-ahead bias
- compare strategy variants fairly

### Initial research choices
For signal research, I want to compare several momentum windows:
- 60 trading days
- 120 trading days
- 180 trading days
- 252 trading days

I will keep the following fixed for now:
- moving average filter = 200 trading days
- volatility window = 60 trading days
- rebalance frequency = monthly

### What I can change later
- top K can later be tested as 2 or 3
- the volatility window can later be changed
- the moving average filter can later be compared with other trend filters

## 6. Compute daily returns

In this section, I compute daily percentage returns from the daily adjusted close prices.

### What this section is doing
Daily returns are needed for:
- rolling volatility estimates
- some later robustness checks
- understanding short-term asset behavior

### Why this matters
Even though the portfolio will probably rebalance monthly, daily returns are still useful because:
- the 200-day moving average is based on daily prices
- volatility targeting is often estimated using daily returns

### What I can change later
- I can use log returns instead of simple returns
- I can later compute weekly returns if I want another rebalance frequency

### What to expect
The first row will be missing and will be dropped. The final dataframe should contain daily return series for all ETFs.

In [5]:
daily_returns = daily_adj_close.pct_change().dropna()

print("Daily returns shape:", daily_returns.shape)
display(daily_returns.head())

Daily returns shape: (5331, 8)


,EEM,EFA,GLD,IEF,IWM,QQQ,SPY,TLT
Date,,,,,,,,
2005-01-04,-0.030789,-0.019170,-0.006509,-0.006226,-0.021436,-0.018228,-0.012220,-0.010480
2005-01-05,-0.012242,-0.000641,-0.001638,0.001654,-0.019987,-0.006189,-0.006901,0.005352
2005-01-06,-0.000680,0.000000,-0.012186,0.000945,0.005302,-0.004929,0.005084,0.000680
2005-01-07,0.001936,-0.004489,-0.007355,-0.000825,-0.011117,0.005215,-0.001433,0.002264
2005-01-10,0.001253,0.004509,0.002629,-0.000119,0.010176,-0.000518,0.004728,0.001582


## 7. Compute momentum signals

In this section, I compute candidate momentum signals for several lookback windows.

### What this section is doing
For each ETF, I calculate trailing returns over different horizons:
- 60 trading days
- 120 trading days
- 180 trading days
- 252 trading days

These momentum scores will later be used to rank ETFs at each monthly decision date.

### Why this matters
Momentum is one of the main selection signals in the strategy. It tells me which ETFs have been strongest recently.

### Why I am testing multiple lookbacks
I do not want to assume one lookback is best from the start. Different horizons may behave differently:
- shorter windows may react faster
- longer windows may be more stable

### What I can change later
- I can add more windows
- I can compute momentum using monthly prices instead
- I can build weighted momentum signals later

### What to expect
This section will create one momentum dataframe for each lookback window.

In [6]:
MOMENTUM_WINDOWS = [60, 120, 180, 252]

momentum_signals = {}

for window in MOMENTUM_WINDOWS:
    momentum = daily_adj_close.pct_change(window)
    momentum_signals[window] = momentum

print("Momentum signal windows created:", list(momentum_signals.keys()))
display(momentum_signals[60].head())

Momentum signal windows created: [60, 120, 180, 252]


,EEM,EFA,GLD,IEF,IWM,QQQ,SPY,TLT
Date,,,,,,,,
2005-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-01-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 8. Align momentum to monthly decision dates

In this section, I convert the daily momentum signals into month-end decision signals.

### What this section is doing
Because the strategy will rebalance monthly, I only need the last available momentum observation for each completed month.

This section resamples each daily momentum dataframe to business month-end and takes the last value.

### Why this matters
All strategy inputs need to line up on the same monthly decision dates. If momentum is daily but the moving average filter and portfolio construction are monthly, then the signals must be aligned before they can be combined correctly.

### What I can change later
- If I move to weekly rebalancing later, I would align to weekly dates instead
- If I want to experiment with monthly-price momentum instead of daily lookback momentum, I can compare both approaches

### What to expect
I should get one monthly momentum dataframe for each lookback window, with the same general monthly index structure as the rebalance schedule.

In [7]:
monthly_momentum_signals = {}

for window, df in momentum_signals.items():
    monthly_signal = df.resample("BM").last()

    # Drop incomplete final month if needed
    if len(monthly_signal) > 0 and daily_adj_close.index.max().date() < monthly_signal.index.max().date():
        monthly_signal = monthly_signal.iloc[:-1]

    monthly_momentum_signals[window] = monthly_signal

for window in MOMENTUM_WINDOWS:
    print(f"Window {window}: shape = {monthly_momentum_signals[window].shape}")

display(monthly_momentum_signals[60].head())

Window 60: shape = (254, 8)
Window 120: shape = (254, 8)
Window 180: shape = (254, 8)
Window 252: shape = (254, 8)


/var/folders/sx/02p4fzbd32gghvv7j9kwgb5m0000gn/T/ipykernel_74251/3755978997.py:4: FutureWarning: 'BM' is deprecated and will be removed in a future version, please use 'BME' instead.
  monthly_signal = df.resample("BM").last()
/var/folders/sx/02p4fzbd32gghvv7j9kwgb5m0000gn/T/ipykernel_74251/3755978997.py:4: FutureWarning: 'BM' is deprecated and will be removed in a future version, please use 'BME' instead.
  monthly_signal = df.resample("BM").last()


,EEM,EFA,GLD,IEF,IWM,QQQ,SPY,TLT
Date,,,,,,,,
2005-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-02-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-03-31,0.015269,-0.001445,-0.004649,-0.011086,-0.040054,-0.074177,-0.015600,0.013625
2005-04-29,-0.014128,-0.013257,0.028226,0.007635,-0.080474,-0.069186,-0.025701,0.017985
2005-05-31,-0.062613,-0.065158,-0.039880,0.035841,-0.042329,0.014925,-0.022657,0.057273


## 9. Compute the 200-day moving average

In this section, I calculate the 200-day moving average for each ETF.

### What this section is doing
For each ETF, I compute the rolling 200-day average of daily adjusted close prices.

### Why this matters
The 200-day moving average acts as a long-term trend filter. In the baseline strategy, an ETF will only be considered eligible if its price is above its 200-day moving average.

### Why this is useful
This helps reduce the chance of holding assets that still rank well on momentum but have already broken down in their broader trend.

### What I can change later
- I can compare 150-day or 250-day moving averages later
- I can test other trend filters in future notebooks

### What to expect
This section will create:
- the 200-day moving average values
- a boolean pass/fail signal showing whether price is above the moving average

In [8]:
ma_200 = daily_adj_close.rolling(window=200).mean()
ma_filter_daily = daily_adj_close > ma_200

display(ma_200.head())
display(ma_filter_daily.head())

,EEM,EFA,GLD,IEF,IWM,QQQ,SPY,TLT
Date,,,,,,,,
2005-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-01-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,EEM,EFA,GLD,IEF,IWM,QQQ,SPY,TLT
Date,,,,,,,,
2005-01-03,False,False,False,False,False,False,False,False
2005-01-04,False,False,False,False,False,False,False,False
2005-01-05,False,False,False,False,False,False,False,False
2005-01-06,False,False,False,False,False,False,False,False
2005-01-07,False,False,False,False,False,False,False,False


## 10. Align the moving average filter to monthly dates

In this section, I convert the daily moving average pass/fail signal into a monthly decision signal.

### What this section is doing
At each month-end decision date, I keep the last available moving average filter value for that month.

### Why this matters
The strategy rebalances monthly, so I only need to know whether each ETF passed the filter at the monthly decision point.

### What I can change later
- If I later use weekly rebalancing, I would align this filter to weekly decision dates instead

### What to expect
The result should be a monthly boolean dataframe where:
- `True` means the ETF is above its 200-day moving average
- `False` means it is below

In [9]:
ma_filter_monthly = ma_filter_daily.resample("BM").last()

if len(ma_filter_monthly) > 0 and daily_adj_close.index.max().date() < ma_filter_monthly.index.max().date():
    ma_filter_monthly = ma_filter_monthly.iloc[:-1]

print("Monthly MA filter shape:", ma_filter_monthly.shape)
display(ma_filter_monthly.head())

Monthly MA filter shape: (254, 8)


/var/folders/sx/02p4fzbd32gghvv7j9kwgb5m0000gn/T/ipykernel_74251/2836412534.py:1: FutureWarning: 'BM' is deprecated and will be removed in a future version, please use 'BME' instead.
  ma_filter_monthly = ma_filter_daily.resample("BM").last()


,EEM,EFA,GLD,IEF,IWM,QQQ,SPY,TLT
Date,,,,,,,,
2005-01-31,False,False,False,False,False,False,False,False
2005-02-28,False,False,False,False,False,False,False,False
2005-03-31,False,False,False,False,False,False,False,False
2005-04-29,False,False,False,False,False,False,False,False
2005-05-31,False,False,False,False,False,False,False,False


## 11. Compute rolling volatility

In this section, I calculate rolling annualized volatility using daily returns.

### What this section is doing
I estimate trailing volatility using a 60-trading-day rolling window.

Then I annualize the result by multiplying by the square root of 252.

### Why this matters
This signal will later support volatility targeting or inverse-volatility weighting.

### Why 60 days for now
A 60-day volatility window is a reasonable balance:
- short enough to respond to changing risk
- long enough to avoid being too noisy

### What I can change later
- I can compare 20-day, 90-day, or 120-day windows
- I can use exponential weighting later instead of simple rolling volatility

### What to expect
The result should be a daily dataframe of annualized volatility estimates for each ETF.

In [10]:
VOL_WINDOW = 60

rolling_vol_daily = daily_returns.rolling(VOL_WINDOW).std() * np.sqrt(252)

print("Rolling daily volatility shape:", rolling_vol_daily.shape)
display(rolling_vol_daily.head())

Rolling daily volatility shape: (5331, 8)


,EEM,EFA,GLD,IEF,IWM,QQQ,SPY,TLT
Date,,,,,,,,
2005-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-01-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-01-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 12. Align volatility to monthly dates

In this section, I convert the daily rolling volatility estimates into monthly decision values.

### What this section is doing
At each month-end decision date, I keep the last available volatility estimate for each ETF.

### Why this matters
If the strategy later uses monthly rebalancing, then the volatility estimate must be aligned to the same monthly decision schedule.

### What I can change later
- I can later compare different volatility windows
- I can also compare annualized volatility with non-annualized volatility if needed

### What to expect
The result should be a monthly volatility dataframe with one value per ETF per monthly decision date.

In [11]:
rolling_vol_monthly = rolling_vol_daily.resample("BM").last()

if len(rolling_vol_monthly) > 0 and daily_returns.index.max().date() < rolling_vol_monthly.index.max().date():
    rolling_vol_monthly = rolling_vol_monthly.iloc[:-1]

print("Monthly rolling volatility shape:", rolling_vol_monthly.shape)
display(rolling_vol_monthly.head())

Monthly rolling volatility shape: (254, 8)


/var/folders/sx/02p4fzbd32gghvv7j9kwgb5m0000gn/T/ipykernel_74251/3190479680.py:1: FutureWarning: 'BM' is deprecated and will be removed in a future version, please use 'BME' instead.
  rolling_vol_monthly = rolling_vol_daily.resample("BM").last()


,EEM,EFA,GLD,IEF,IWM,QQQ,SPY,TLT
Date,,,,,,,,
2005-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-02-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005-03-31,0.182039,0.102975,0.113405,0.049969,0.149167,0.148495,0.104125,0.100043
2005-04-29,0.209880,0.119696,0.109681,0.056621,0.169413,0.157173,0.124185,0.104920
2005-05-31,0.218946,0.118358,0.098975,0.054345,0.167210,0.145390,0.120474,0.096160


## 13. Align all monthly signals

In this section, I make sure the monthly signal dataframes all use the same index.

### What this section is doing
I find the common monthly dates that exist in:
- the momentum tables
- the moving average filter table
- the rolling volatility table

Then I align everything to that shared monthly index.

### Why this matters
If the signals do not line up on the same decision dates, later portfolio construction can become inconsistent or accidentally use mismatched information.

### What I can change later
- If I later switch the signal frequency, I should repeat this alignment step for the new schedule

### What to expect
After this section, all signals should have the same monthly index.

In [12]:
common_index = ma_filter_monthly.index.intersection(rolling_vol_monthly.index)

for window in MOMENTUM_WINDOWS:
    common_index = common_index.intersection(monthly_momentum_signals[window].index)

common_index = common_index.sort_values()

# Keep only dates where all required signals are available
valid_dates = []

for dt in common_index:
    all_momentum_available = all(
        monthly_momentum_signals[window].loc[dt].notna().all()
        for window in MOMENTUM_WINDOWS
    )
    vol_available = rolling_vol_monthly.loc[dt].notna().all()

    # MA filter is boolean, but I still want the moving average history to be mature.
    # Since the MA filter comes from a 200-day rolling average, early rows are not reliable.
    # A simple practical rule is to require that volatility and all momentum windows are available.
    if all_momentum_available and vol_available:
        valid_dates.append(dt)

common_index = pd.Index(valid_dates)

ma_filter_monthly = ma_filter_monthly.loc[common_index]
rolling_vol_monthly = rolling_vol_monthly.loc[common_index]

for window in MOMENTUM_WINDOWS:
    monthly_momentum_signals[window] = monthly_momentum_signals[window].loc[common_index]

print("Common monthly index length:", len(common_index))
print("First usable signal date:", common_index.min())
print("Last usable signal date:", common_index.max())

Common monthly index length: 254
First date: 2005-01-31 00:00:00
Last date: 2026-02-27 00:00:00


## 14. Inspect sample dates

In this section, I inspect a few example monthly decision dates to make sure the signals look reasonable.

### What this section is doing
For selected months, I display:
- momentum rankings
- moving average pass/fail status
- rolling volatility estimates

### Why this matters
Before backtesting anything, I want to verify that the signal logic behaves sensibly on real dates.

### What I can change later
- I can inspect different dates, especially around major market events
- I can later automate a top-K selection table from these signals

### What to expect
The output should help me answer:
- which ETFs look strongest?
- which ones pass the trend filter?
- which ones are more volatile?

In [13]:
sample_dates = [
    common_index[50],
    common_index[100],
    common_index[150],
]

for dt in sample_dates:
    print("=" * 80)
    print("Decision date:", dt.date())

    sample_table = pd.DataFrame({
        "Momentum_60": monthly_momentum_signals[60].loc[dt],
        "Momentum_120": monthly_momentum_signals[120].loc[dt],
        "Momentum_180": monthly_momentum_signals[180].loc[dt],
        "Momentum_252": monthly_momentum_signals[252].loc[dt],
        "Above_200DMA": ma_filter_monthly.loc[dt],
        "RollingVol_60d": rolling_vol_monthly.loc[dt]
    }).sort_values("Momentum_120", ascending=False)

    display(sample_table)

Decision date: 2009-03-31


,Momentum_60,Momentum_120,Momentum_180,Momentum_252,Above_200DMA,RollingVol_60d
TLT,-0.086416,0.086670,0.167618,0.168287,True,0.233432
IEF,-0.000010,0.081316,0.121417,0.105792,True,0.118139
GLD,0.046967,0.034491,-0.058701,0.039374,True,0.285063
EEM,-0.051605,-0.039722,-0.414446,-0.452897,False,0.573187
QQQ,-0.021265,-0.068477,-0.311899,-0.331960,False,0.383224
SPY,-0.138456,-0.192849,-0.338277,-0.402585,False,0.407850
EFA,-0.172755,-0.204757,-0.419337,-0.477029,False,0.472435
IWM,-0.159954,-0.238201,-0.356130,-0.395996,False,0.509432


Decision date: 2013-05-31


,Momentum_60,Momentum_120,Momentum_180,Momentum_252,Above_200DMA,RollingVol_60d
IWM,0.061073,0.204633,0.183955,0.284471,True,0.162808
SPY,0.062634,0.164540,0.158274,0.249703,True,0.112409
QQQ,0.071661,0.130258,0.080230,0.180546,True,0.122322
EFA,0.016757,0.094787,0.149278,0.275767,True,0.138183
IEF,-0.009442,-0.027098,-0.014610,-0.000513,False,0.054857
EEM,-0.055696,-0.028056,0.033477,0.098413,False,0.147268
TLT,-0.016410,-0.078755,-0.065633,-0.047666,False,0.139883
GLD,-0.126020,-0.185897,-0.199474,-0.113230,False,0.268076


Decision date: 2017-07-31


,Momentum_60,Momentum_120,Momentum_180,Momentum_252,Above_200DMA,RollingVol_60d
EEM,0.103937,0.174351,0.227754,0.229110,True,0.124297
EFA,0.048459,0.140211,0.193829,0.183956,True,0.081054
QQQ,0.048017,0.139590,0.226898,0.255802,True,0.131427
SPY,0.038589,0.087849,0.157779,0.159661,True,0.071151
IWM,0.029118,0.053908,0.167933,0.185380,True,0.116621
TLT,0.027890,0.039179,0.012921,-0.102077,True,0.087499
GLD,0.033907,0.028010,-0.006663,-0.063808,True,0.093390
IEF,0.011355,0.020323,0.001310,-0.038375,True,0.038611


## 15. Build an initial top-K candidate table

In this section, I create a first-pass selection table using one momentum window and the moving average filter.

### What this section is doing
As an example, I use the 120-day momentum signal and:
- keep only ETFs above the 200-day moving average
- rank them by momentum
- record the top 2 candidates

This is not yet the full backtest. It is just a signal-level preview of what the strategy would have wanted to hold.

### Why this matters
This gives me an interpretable check before moving to full portfolio simulation.

### What I can change later
- I can use 60, 180, or 252-day momentum instead
- I can change top 2 to top 3
- I can later add volatility-aware weighting in the backtest notebook

### What to expect
I should get a monthly table showing the selected ETFs for each decision date.

In [14]:
TOP_K = 2
BASELINE_WINDOW = 120

selection_records = []

for dt in common_index:
    momentum_today = monthly_momentum_signals[BASELINE_WINDOW].loc[dt]
    ma_today = ma_filter_monthly.loc[dt]

    eligible = momentum_today[ma_today].dropna().sort_values(ascending=False)
    selected = eligible.head(TOP_K).index.tolist()

    record = {
        "Date": dt,
        "NumEligible": len(eligible),
        "Selected_1": selected[0] if len(selected) > 0 else None,
        "Selected_2": selected[1] if len(selected) > 1 else None,
    }
    selection_records.append(record)

selection_table = pd.DataFrame(selection_records).set_index("Date")
display(selection_table.head(10))

,NumEligible,Selected_1,Selected_2
Date,,,
2005-01-31,0,None,None
2005-02-28,0,None,None
2005-03-31,0,None,None
2005-04-29,0,None,None
2005-05-31,0,None,None
2005-06-30,0,None,None
2005-07-29,0,None,None
2005-08-31,0,None,None
2005-09-30,0,None,None


## 16. Split the signal tables into train / validation / test

In this section, I apply the research split to the signal data so that later notebooks can evaluate strategies more rigorously.

### What this section is doing
I split the selection table and key signal tables into:
- train
- validation
- test

### Why this matters
This makes the signal research pipeline consistent with the evaluation framework that will be used later.

### What I can change later
- I can later create helper functions in `src/` so that splitting is reusable across notebooks

### What to expect
I should see the number of monthly observations in each period.

In [15]:
selection_train, selection_valid, selection_test = split_by_date(
    selection_table, TRAIN_START, TRAIN_END, VALID_START, VALID_END, TEST_START, TEST_END
)

print("Selection table observations")
print("Train:", len(selection_train))
print("Validation:", len(selection_valid))
print("Test:", len(selection_test))

Selection table observations
Train: 120
Validation: 48
Test: 86


## 17. Save signal artifacts for later notebooks

In this section, I save the signal tables so that later backtesting notebooks can load them directly.

### What this section is doing
I save:
- monthly moving average filter
- monthly rolling volatility
- monthly momentum tables for each window
- the initial selection table

### Why this matters
Saving these files makes later notebooks cleaner and more reproducible.

### What I can change later
- I can save to Parquet for faster reading
- I can save additional signal tables later as the project grows

### What to expect
This section should create several CSV files in the processed data folder.

In [16]:
ma_filter_monthly.to_csv(PROCESSED_DIR / "ma_filter_monthly.csv")
rolling_vol_monthly.to_csv(PROCESSED_DIR / "rolling_vol_monthly.csv")
selection_table.to_csv(PROCESSED_DIR / "selection_table_baseline_120d.csv")

for window in MOMENTUM_WINDOWS:
    monthly_momentum_signals[window].to_csv(PROCESSED_DIR / f"momentum_{window}d_monthly.csv")

print("Saved signal artifacts to:", PROCESSED_DIR.resolve())

Saved signal artifacts to: /Users/nicholasturangan/Desktop/quant/quant-portfolio-project/data/processed


## 18. Key takeaways from this notebook

At this stage, I have transformed the cleaned ETF price data into the main decision signals for the strategy.

### What I accomplished
- loaded the processed daily and monthly datasets
- defined the train / validation / test framework
- computed candidate momentum signals
- computed the 200-day moving average filter
- computed rolling volatility estimates
- aligned all signals to monthly decision dates
- built an initial top-K selection table
- saved signal artifacts for later notebooks

### What I learned
This notebook shows how the strategy would identify strong ETFs and screen them using a long-term trend filter. It also confirms that all signals can be aligned consistently on a monthly decision schedule.

### What comes next
The next notebook should build the first full backtest using a shared baseline strategy. That notebook will answer questions such as:
- how the signal-based portfolio would have performed
- how different momentum windows compare
- whether the baseline strategy is promising enough to justify deeper tuning